# 02 — XGBoost: five- and ten-minute forecasts
Requires [00 — Data preparation](00_Data_Preparation.ipynb). Sends its predictions to [05 — Combined results](05_Combined_Results.ipynb).

Both direct and recursive approaches are tested separately at each horizon.

Each setup uses 30 Optuna trials with the earlier parameter ranges, source weights and trial row caps. Test rows never select settings. These test groups have been examined previously, so this is a rerun, not fresh independent confirmation.

## 1. Setup and load the common prepared data

In [ ]:
%pip -q install "scikit-learn==1.6.1" "optuna>=4,<5" "xgboost>=2,<4" joblib

In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/KC_CO2_Modular_Notebooks")
sys.path.insert(0, str(PROJECT))
RUN_DIR = PROJECT / "runs" / "real_sessions_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from kc_common import *


In [ ]:
MODEL_NAME = "XGBoost"
bundle = joblib.load(RUN_DIR / "prepared.joblib")
DATA, STATE, HIST = bundle["DATA"], bundle["STATE"], bundle["HIST"]
run_id = bundle["run_id"]
OUT = RUN_DIR / MODEL_NAME
OUT.mkdir(exist_ok=True)
RESULTS, MODELS, STUDIES, TIMINGS, PREDICTIONS = [], {}, {}, {}, []
np.random.seed(SEED)
random.seed(SEED)


## 2. Evaluation and output helpers
MAE and RMSE are in ppm; R² has no unit. Source = ALL means ordinary row-pooled results, without source balancing.

In [ ]:
def add_metrics(label, horizon, approach, split, ids, pred):
    record_predictions(label, horizon, approach, split, ids, pred)
    d = DATA[horizon].iloc[ids]
    actual = d.actual_future.to_numpy()
    for source in ["ALL", *SOURCE_SHARE]:
        mask = np.ones(len(d), dtype=bool) if source == "ALL" else d.source.to_numpy() == source
        a = actual[mask]
        p = np.asarray(pred)[mask]
        RESULTS.append(
            {
                "Model": label,
                "Horizon_min": horizon,
                "Approach": approach,
                "Split": split,
                "Source": source,
                "Rows": len(a),
                "MAE": mean_absolute_error(a, p),
                "RMSE": mean_squared_error(a, p) ** 0.5,
                "R2": r2_score(a, p),
            }
        )


def record_predictions(label, horizon, approach, split, ids, pred):
    d = DATA[horizon].iloc[ids]
    frame = d[["source", "session", "time", "actual_future", "CO2_now"]].copy()
    frame["row_id"] = ids
    frame["predicted_future"] = pred
    frame["Model"] = label
    frame["Horizon_min"] = horizon
    frame["Approach"] = approach
    frame["Split"] = split
    frame["run_id"] = run_id
    PREDICTIONS.append(frame)


def save_outputs():
    pd.DataFrame(RESULTS).to_csv(OUT / "metrics.csv", index=False)
    pd.concat(PREDICTIONS).to_csv(OUT / "predictions.csv.gz", index=False)
    for key, study in STUDIES.items():
        name = "_".join(map(str, key))
        study.trials_dataframe().to_csv(OUT / f"{name}_trials.csv", index=False)
        params = dict(study.best_params)
        params.update(study.best_trial.user_attrs)
        (OUT / f"{name}_settings.json").write_text(json.dumps(params, indent=2))
    pd.DataFrame(
        [
            {
                "Model": k[0],
                "Horizon_min": k[1],
                "Approach": k[2],
                "Study_and_refit_minutes": t / 60,
            }
            for k, t in TIMINGS.items()
        ]
    ).to_csv(OUT / "timings.csv", index=False)
    (OUT / "run_id.txt").write_text(run_id)
    import importlib.metadata

    versions = {
        name: importlib.metadata.version(name)
        for name in ["numpy", "pandas", "scikit-learn", "optuna", "xgboost", "joblib"]
    }
    (OUT / "versions.json").write_text(json.dumps(versions, indent=2))


## 3. Model and tuning helpers
Direct models learn the change at the requested horizon. Recursive models learn a one-minute change; each trial is scored after the full five- or ten-step rollout. Future temperature, humidity and occupancy are held at their last known values. Final fits use all training rows.

In [ ]:
def tree_model(kind, trial=None, params=None):
    if params is None:
        if kind == "RF":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 80, 240, step=40),
                "max_depth": trial.suggest_int("max_depth", 3, 14),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 15),
                "max_features": trial.suggest_float("max_features", 0.4, 1.0),
            }
        else:
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
                "max_depth": trial.suggest_int("max_depth", 2, 7),
                "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.15, log=True),
                "subsample": trial.suggest_float("subsample", 0.65, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10, log=True),
            }
    if kind == "RF":
        return RandomForestRegressor(**params, n_jobs=-1, random_state=SEED)
    return XGBRegressor(
        **params,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=4,
        random_state=SEED,
        verbosity=0
    )


def tree_predict(model, kind, horizon, approach, ids):
    d = DATA[horizon]
    st = STATE[horizon]
    if approach == "direct":
        return d.CO2_now.to_numpy()[ids] + model.predict(st["X"][ids])
    c = HIST[horizon]["c"][ids].copy()
    t = HIST[horizon]["t"][ids].copy()
    h = HIST[horizon]["h"][ids].copy()
    initial = HIST[horizon]["initial"][ids]
    elapsed = HIST[horizon]["elapsed"][ids]
    g = d.iloc[ids]
    for step in range(horizon):
        f = make_features(
            c,
            t,
            h,
            initial,
            elapsed + step,
            g.people.to_numpy(),
            g.time + pd.Timedelta(minutes=step),
            g.room.to_numpy(),
        )
        delta = model.predict(st["prep"].transform(f[FEATURES]).astype(np.float32))
        c = np.column_stack([c[:, 1:], c[:, -1] + delta])
        t = np.column_stack([t[:, 1:], t[:, -1]])
        h = np.column_stack([h[:, 1:], h[:, -1]])
    return c[:, -1]


def run_tree(kind, horizon, approach):
    key = (kind, horizon, approach)
    if key in MODELS:
        return
    d = DATA[horizon]
    st = STATE[horizon]
    fitids = st["trial_train"]
    valids = st["trial_val"]
    target = "target_direct" if approach == "direct" else "target_recursive"
    start = time.time()

    def objective(trial):
        model = tree_model(kind, trial=trial)
        w = balanced_weights(d.source.iloc[fitids])
        w *= len(w) / w.sum()
        model.fit(st["X"][fitids], d[target].to_numpy()[fitids], sample_weight=w)
        pred = tree_predict(model, kind, horizon, approach, valids)
        v = d.iloc[valids]
        return score_sources(v.actual_future.to_numpy(), pred, v.source.to_numpy())

    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    STUDIES[key] = study
    final = tree_model(kind, params=study.best_params)
    ids = st["train"]
    w = balanced_weights(d.source.iloc[ids])
    w *= len(w) / w.sum()
    final.fit(st["X"][ids], d[target].to_numpy()[ids], sample_weight=w)
    MODELS[key] = final
    TIMINGS[key] = time.time() - start
    for split in ("val", "test"):
        ids = st[split]
        pred = tree_predict(final, kind, horizon, approach, ids)
        add_metrics(
            kind, horizon, approach, {"val": "Validation", "test": "Test"}[split], ids, pred
        )
    print(
        key,
        "best trial score",
        round(study.best_value, 3),
        "parameters",
        study.best_params,
        "elapsed minutes",
        round(TIMINGS[key] / 60, 1),
    )


## 4. Run 10-minute forecasts
Print Local, UPC, Slovenia and ALL scores for validation and test. Results and trial histories are saved after this horizon finishes.

In [ ]:
for approach in ("direct", "recursive"):
    run_tree(MODEL_NAME, 10, approach)
save_outputs()
display(
    pd.DataFrame(RESULTS)
    .query("Horizon_min == 10")
    .sort_values(["Split", "Approach", "Source"])
    .round(3)
)
for approach in ("direct", "recursive"):
    joblib.dump(
        {
            "model": MODELS[(MODEL_NAME, 10, approach)],
            "preprocessor": STATE[10]["prep"],
            "run_id": run_id,
        },
        OUT / f"10min_{approach}.joblib",
        compress=3,
    )


## 5. Run 5-minute forecasts
Print Local, UPC, Slovenia and ALL scores for validation and test. Results and trial histories are saved after this horizon finishes.

In [ ]:
for approach in ("direct", "recursive"):
    run_tree(MODEL_NAME, 5, approach)
save_outputs()
display(
    pd.DataFrame(RESULTS)
    .query("Horizon_min == 5")
    .sort_values(["Split", "Approach", "Source"])
    .round(3)
)
for approach in ("direct", "recursive"):
    joblib.dump(
        {
            "model": MODELS[(MODEL_NAME, 5, approach)],
            "preprocessor": STATE[5]["prep"],
            "run_id": run_id,
        },
        OUT / f"5min_{approach}.joblib",
        compress=3,
    )


## 6. Overall feature importance
Permutation importance follows the earlier comparison: shuffle each input on the source-balanced validation sample and measure the increase in weighted MAE. This is one overall graph across sources, not separate session graphs. Correlated inputs can share importance; the graph does not show cause and effect.

In [ ]:
from sklearn.inspection import permutation_importance


class DeltaWrapper:
    def __init__(self, preprocessor, model):
        self.preprocessor = preprocessor
        self.model = model

    def fit(self, *args, **kwargs):
        return self

    def predict(self, features):
        matrix = self.preprocessor.transform(features[FEATURES]).astype(np.float32)
        return self.model.predict(matrix)


ids = STATE[5]["trial_val"]
frame = DATA[5].iloc[ids]
weights = balanced_weights(frame.source)
wrapped = DeltaWrapper(STATE[5]["prep"], MODELS[(MODEL_NAME, 5, "direct")])


def weighted_neg_mae(estimator, features, target):
    return -np.average(np.abs(target - estimator.predict(features)), weights=weights)


importance = permutation_importance(
    wrapped,
    frame[FEATURES],
    frame.target_direct.to_numpy(),
    scoring=weighted_neg_mae,
    n_repeats=3,
    random_state=SEED,
    n_jobs=1,
)
table = pd.DataFrame(
    {
        "Feature": FEATURES,
        "Mean_increase_MAE": importance.importances_mean,
        "Std_increase_MAE": importance.importances_std,
    }
)
table.to_csv(OUT / "feature_importance.csv", index=False)
top = table.sort_values("Mean_increase_MAE").tail(15)
plt.figure(figsize=(8, 5))
plt.barh(top.Feature, top.Mean_increase_MAE, color="#247ba0")
plt.title(f"{MODEL_NAME}: five-minute direct feature importance")
plt.xlabel("Increase in source-weighted validation MAE after shuffling (ppm)")
plt.tight_layout()
plt.savefig(OUT / "feature_importance.png", dpi=180)
plt.show()


## Next step
Run the other model notebooks, then [05 — Combined results](05_Combined_Results.ipynb). Comparing model tables does not retrain or reweight predictions.